# ODE Integrators I: Explicit Methods

Ordinary differential equations form the mathematical foundation of
physics.
They describe how systems evolve in time, e.g., Newton's second law of
motion is
\begin{align}
  f = m a = m \frac{d^2x}{dt^2},
\end{align}
which is a second-order ODE relating force to acceleration.
More generally, an ODE relates a function of time to its time
derivatives.

While ODEs capture the essence of dynamics, analytic solutions are
possible only in special cases.
Most real systems involve nonlinear forces, damping, or external
driving terms that make exact solutions impossible.
Numerical integration allows us to approximate trajectories step by
step and study systems far beyond what can be solved by hand.

## Problem Definition and Types of ODEs

Consider two forms of first-order ODEs:

1. Time-dependent forcing only:
   \begin{align}
     \frac{dx}{dt} = f(t).
   \end{align}
   Here, the derivative depends only on time.
   Solutions are obtained by direct integration:
   \begin{align}
     x(t) = x(t_0) + \int_{t_0}^{t} f(t') \, dt'.
   \end{align}

2. State- and time-dependent dynamics:
   \begin{align}
     \frac{dx}{dt} = f(x, t).
   \end{align}
   Now, the derivative depends on the state $x$ itself.
   The function we want to solve for also appears inside the RHS,
   making direct integration impossible.
   We cannot evaluate the integral without already knowing $x(t)$ at
   intermediate points.

The first type reduces to standard numerical quadrature (trapezoidal
rule, Simpson's rule, etc.), which we studied earlier.
The second case, general dependence on both $x$ and $t$, is the
typical situation in physics.
Examples include planetary orbits, nonlinear oscillators, chaotic
systems, and interacting biological populations.
In such cases:
* Direct integration is often not feasible; the problem must be solved
  as an initial value problem (IVP).
* Analytic solutions are often unknown or intractable.
* Numerical methods approximate the solution incrementally, using
  small time steps to trace the system's evolution.

By discretizing time and advancing step-by-step, we can model the
behavior of even the most complex systems.
This is the core idea behind numerical ODE integrators.

## Forward Euler Method

Euler's method is the simplest technique for numerically solving
ordinary differential equations.
This method provides an easy way to approximate the solution of an IVP
by advancing one small step at a time.

We can apply Euler's method to an ODE of the form:
\begin{align}
  \frac{dx}{dt} = f(x, t), \quad x(t_0) = x_0
\end{align}
where $x_0$ is the initial value of $x$ at time $t = t_0$.
However, it is usually not recommended in practical calculations
because of its low accuracy and poor stability properties.

There are three simple ways to derive Euler's method.
The easiest way is simply to hold $x$ fixed in $f(x, t)$ and apply the
left Riemann sum.
The left Riemann sum is first order in step size by approximating
$f(x, t)$ as a constant.
In this sense, holding $x$ fixed is somewhat "self-consistent" in terms of
approximation order.

We then recall the definition of a derivative:
\begin{align}
  f(x, t) = \frac{dx}{dt} = \lim_{\Delta t\rightarrow 0}\frac{x(t + \Delta t) - x(t)}{\Delta t}.
\end{align}
If we simply remove the limit and keep the "finite difference" part,
then it is trivial to show
\begin{align}
  x(t + \Delta t) &\approx x(t) + f(x(t), t)\Delta t.
\end{align}
This is, again, the Forward Euler method.
While very intuitive, the above two derivations do not formally show
the order of the Euler method.

We may also consider a numerical approximation to the solution of an
ODE.
We approximate the solution at time $t_{n+1} = t_n + \Delta t$ by
using the Taylor expansion:
\begin{align}
  x(t_{n+1}) = x(t_n) + f(x(t_n), t_n) \Delta t + \mathcal{O}(\Delta t^2)
\end{align}
Neglecting the higher-order terms in the expansion, we obtain once
again the Forward Euler method:
\begin{align}
  x_{n+1} = x_n + f(x_n, t_n) \Delta t
\end{align}
It is thus a step-by-step approach that proceeds by evaluating $f(x,
t)$ at each time point and then advancing to the next point.
It is a first-order explicit method.

### Explicit and Implicit Methods

A method is called *explicit* if the new value $x_{n+1}$ is given
directly in terms of quantities that are already known, i.e.,
\begin{align}
  x_{n+1} = \Phi(x_n, t_n, \Delta t),
\end{align}
so that each step requires only evaluations of $f$ at known states.
In contrast, an *implicit* method has $x_{n+1}$ appearing on both
sides of the update, e.g., the backward Euler method
\begin{align}
  x_{n+1} = x_n + f(x_{n+1}, t_{n+1}) \Delta t,
\end{align}
which requires solving a (generally nonlinear) equation for $x_{n+1}$
at every step.
Explicit methods are cheap and simple per step, but, as we will see,
they can suffer from severe step-size restrictions for stiff problems.

Let's solve the simple differential equation:
\begin{align}
  \frac{dx}{dt} = \lambda x(t)
\end{align}

This equation has solution
\begin{align}
  x(t) = x(t_0)\exp[\lambda(t-t_0)]
\end{align}

If we choose $\lambda = 1$ and $x(0) = 1$, the solution reduces to
$x(t) = \exp(t)$.

In [ ]:
# Let's visualize the solution:

import numpy as np
from matplotlib import pyplot as plt

T = np.linspace(0, 1, 1001)
X = np.exp(T)

plt.plot(T, X, label=r"$\exp(t)$")
plt.xlabel(r"$t$")
plt.ylabel(r"$x(t)$")
plt.legend()

In [ ]:
# Let's implement Euler's method, with history
# For simplicity, f takes only x as an argument, which is sufficient
# for autonomous ODEs such as the ones in this lecture

def Euler(f, x, t, dt, n):
    X = [np.array(x)]
    T = [np.array(t)]
    for _ in range(n):
        X.append(X[-1] + dt * f(X[-1]))
        T.append(T[-1] + dt)
    return np.array(X), np.array(T)

In [ ]:
# Let's test Euler's method

f  = lambda x: x

n  = 10
dt = 1/n

x0 = 1
t0 = 0

X1, T1 = Euler(f, x0, t0, dt, n)

plt.plot(T,  X,        label=r"$\exp(t)$")
plt.plot(T1, X1, "o-", label=r"Forward Euler")
plt.xlabel(r"$t$")
plt.ylabel(r"$x(t)$")
plt.legend()

In [ ]:
# As always, we can study the convergence of the numerical method

def Err(n=100):
    X1, T1 = Euler(f, 1, 0, 1/n, n)
    X = np.exp(T1)
    return np.max(abs(X1 - X))

N  = np.array([32, 64, 128, 256, 512, 1024])
E1 = np.array([Err(n) for n in N])

plt.loglog(N, 1/N, "k--", label=r"$1/n$")
plt.loglog(N, E1,  "o:",  label=r"Forward Euler")
plt.xlabel(r"$n$")
plt.ylabel(r"$\text{err} = \max|x_\text{numeric} - x|$")
plt.legend()

### Higher-Order ODEs and Systems of ODEs

In computational astrophysics, we often encounter systems governed by
Newton's laws:
\begin{align}
  m \frac{d^2 x}{dt^2} = f(x, t)
\end{align}

This equation is a second-order ordinary differential equation because
it involves the second derivative of $x$ with respect to $t$.
However, it is often more practical to convert second-order ODEs into
a system of first-order ODEs.
To do this, we introduce a new variable, $v = dx/dt$, representing the
velocity of the object.
This substitution allows us to rewrite the second-order ODE as two
coupled first-order ODEs:
\begin{align}
  \frac{dx}{dt} &= v \\
  \frac{dv}{dt} &= \frac{1}{m}f(x, t)
\end{align}

This formulation is convenient for numerical methods,
which are generally well-suited to solving systems of first-order
ODEs.
To further simplify, we can express these equations in vector notation
by defining $\mathbf{x} = [x, v]^T$ and $\mathbf{f} = [v, f/m]^T$.
The system then becomes:
\begin{align}
  \frac{d\mathbf{x}}{dt} = \mathbf{f}(\mathbf{x}, t).
\end{align}
This vector form emphasizes the structure of the system and enables us
to apply general numerical techniques to solve both equations
simultaneously.

To illustrate this approach, let's consider a classic example: the
simple pendulum under gravity.
The motion of a pendulum of length $l$, swinging under gravity $g$,
can be described by the second-order equation:
\begin{align}
  \frac{d^2\theta}{dt^2} + \frac{g}{l} \sin\theta = 0
\end{align}

Here, $\theta(t)$ is the angle of the pendulum with the vertical, and
the term $\sin \theta$ introduces nonlinearity, which makes the
equation challenging to solve analytically.
Converting this equation into a system of first-order ODEs allows us
to handle it more effectively with numerical methods.
We define $\Omega = \frac{d\theta}{dt}$, the angular velocity, leading
to the following system:
\begin{align}
  \frac{d\theta(t)}{dt} &= \Omega(t)\\
  \frac{d\Omega(t)}{dt} &= - \frac{g}{l}\sin\theta(t)
\end{align}

In vector notation, we represent the system as:
\begin{align}
  \frac{d\mathbf{x}(t)}{dt} = \mathbf{f}(\mathbf{x}, t)
\end{align}
where
\begin{align}
  \mathbf{x} &= \begin{bmatrix} \theta(t) \\ \Omega(t) \end{bmatrix}, \\
  \mathbf{f}(\mathbf{x}, t) &= \begin{bmatrix} \Omega(t) \\ -\frac{g}{l} \sin \theta(t) \end{bmatrix}.
\end{align}

Later in the lecture, we will try to solve this full problem.
But to derive and compare different numerical methods, let's first
reduce the problem to something that has an analytical solution.
Specifically, we can simplify the pendulum problem further by assuming
small oscillations, where the angle $\theta$ is small enough that
$\sin \theta \approx \theta$.
This approximation linearizes the equation of motion, reducing the
system to a simple harmonic oscillator.

In this approximation, the equation of motion becomes:
\begin{align}
  \frac{d^2 \theta}{dt^2} + \frac{g}{l} \theta = 0
\end{align}
As a result, the system of ODEs becomes:
\begin{align}
  \mathbf{x} &= \begin{bmatrix} \theta(t) \\ \Omega(t) \end{bmatrix}, \\
  \mathbf{f}(\mathbf{x}, t) &= \begin{bmatrix} \Omega(t) \\ -\frac{g}{l} \theta(t) \end{bmatrix}.
\end{align}

With initial conditions $\theta(0) = \theta_0$ and $\Omega(0) = \Omega_0$,
this system has the analytical solution
\begin{align}
  \theta(t) &= \theta_0 \cos(\omega_0 t) + \frac{\Omega_0}{\omega_0} \sin(\omega_0 t), \\
  \Omega(t) &= -\theta_0 \omega_0 \sin(\omega_0 t) + \Omega_0 \cos(\omega_0 t),
\end{align}
where $\omega_0 = \sqrt{g/l}$ is the natural angular frequency of the
pendulum.

In [ ]:
# Let's first plot the analytical solution
# We choose g = l = 1 so that omega_0 = sqrt(g/l) = 1, and initial
# conditions theta_0 = 0 and Omega_0 = 0.01, so theta(t) = 0.01 sin(t)

T     = np.linspace(0, 10, 101)
Theta = 0.01 * np.sin(T)

plt.plot(T, Theta, label=r"$0.01\sin(t)$")
plt.xlabel(r"$t$")
plt.ylabel(r"$\theta(t)$")
plt.legend()

In [ ]:
# Thanks to operator overloading in NumPy, our Forward Euler method
# already works for systems of ODEs, as long as f returns a NumPy array.
# There is no need to rewrite it!
#
# Compare the analytical and numerical solutions

def F(x):
    theta, omega = x
    return np.array([omega, -theta])

n  = 64
dt = 10 / n

X0 = [0, 0.01]
t0 = 0

X1, T1 = Euler(F, X0, t0, dt, n)

Theta1 = X1[:,0]
Omega1 = X1[:,1]

plt.plot(T,  Theta,        label=r"$0.01\sin(t)$")
plt.plot(T1, Theta1, "o-", label=r"Forward Euler")
plt.xlabel(r"$t$")
plt.ylabel(r"$\theta(t)$")
plt.legend()

In [ ]:
# Again, we can study the convergence of the numerical method

def Err(integrator, n=100):
    Xf, Tf = integrator(F, [0, 0.01], 0, 10/n, n)
    Thetaf = Xf[:,0]
    Theta  = 0.01 * np.sin(Tf)
    return np.max(abs(Thetaf - Theta))

N  = np.array([64, 128, 256, 512, 1024])
E1 = np.array([Err(Euler, n) for n in N])

plt.loglog(N, 1/N, "k--", label=r"$1/n$")
plt.loglog(N, E1,  "o:",  label=r"Forward Euler")
plt.xlabel(r"$n$")
plt.ylabel(r"$\text{err} = \max|x_\text{numeric} - x|$")
plt.legend()

## Second-Order Runge-Kutta Method

The Forward Euler method is only first-order accurate, meaning its
error decreases linearly with the step size $\Delta t$, as we saw in
the convergence plots.
While it is simple to implement, this method's convergence rate is
too slow.
Achieving higher accuracy requires very small steps, which can become
computationally expensive.

This naturally raises the question: can we improve the convergence
rate of our numerical solution, reducing the error more rapidly as we
use smaller steps?

To explore this, we can draw inspiration from our previous work on
numerical integration.
In that context, we observed that the midpoint (or central) Riemann
sum converges faster than the left or right Riemann sums.
This suggests that a midpoint approach may also provide advantages in
solving ODEs.

One possible improvement is to propose a midpoint method that attempts
to evaluate the function $\mathbf{f}$ at the midpoint between steps.
Mathematically, this approach can be expressed as:
\begin{align}
  \mathbf{x}_{n+1} = \mathbf{x}_n + \mathbf{f}(\mathbf{x}_{n+1/2}, t_{n+1/2}) \Delta t
\end{align}

However, a significant problem arises: the midpoint value
$\mathbf{f}_{n+1/2} \equiv \mathbf{f}(\mathbf{x}_{n+1/2}, t_{n+1/2})$
is unknown at step $n$.
We need to know the value of $\mathbf{x}$ at the midpoint $t_{n+1/2}$
to use this method, but this value cannot be calculated without
already knowing the future values of $\mathbf{x}$.
This issue makes a straightforward midpoint method impractical for
generic ODEs, where $\mathbf{f}$ depends on both $\mathbf{x}$ and $t$.

An exception occurs if $\mathbf{f}$ depends only on $t$, as in
$d\mathbf{x}/dt = \mathbf{f}(t)$;
in such cases, a true midpoint method is feasible, which is nothing
but our midpoint Riemann sum.
However, for most ODEs, including those where $\mathbf{f}$ depends on
$\mathbf{x}$, a different approach is necessary.

To work around this issue, we can approximate the midpoint value
instead of calculating it exactly.
It suffices to find an approximate solution for $\mathbf{x}$ at the
half-step, which we denote as $\tilde{\mathbf{x}}_{n+1/2}$.

The simplest way to do this is to use the Forward Euler method to
compute an estimated value at the midpoint.
Specifically, we can approximate $\mathbf{x}$ at $t_{n+1/2}$ as:
\begin{align}
  \tilde{\mathbf{x}}_{n+1/2} = \mathbf{x}_n + \mathbf{f}(\mathbf{x}_n, t_n) \frac{\Delta t}{2}
\end{align}
Using this half-step approximation, we then proceed with a full step
to find $\mathbf{x}_{n+1}$ by evaluating $\mathbf{f}$ at the midpoint:
\begin{align}
  \mathbf{x}_{n+1} = \mathbf{x}_n + \mathbf{f}(\tilde{\mathbf{x}}_{n+1/2}, t_{n+1/2}) \Delta t
\end{align}
This approach, known as the second-order Runge-Kutta method,
incorporates midpoint information and achieves second-order accuracy.

This second-order Runge-Kutta method improves convergence by
leveraging an approximate midpoint, resulting in a more accurate
solution than the first-order Euler method without requiring prohibitively
small step sizes.

In [ ]:
# Let's visualize how RK2 builds up the slope for a single step,
# using the simple ODE dx/dt = x with x(0) = 1 from earlier.
# We use a large step size so that each stage is easy to see.

def plot_RK2_step(f, exact, x0, t0, dt):
    tt = np.linspace(t0, t0 + dt, 101)
    th = t0 + 0.5 * dt

    k1 = f(x0)                # slope at the beginning of the step
    xh = x0 + 0.5 * dt * k1   # Forward Euler estimate at the midpoint
    k2 = f(xh)                # slope at the estimated midpoint
    xE = x0 + dt * k1         # Forward Euler full step, for comparison
    x1 = x0 + dt * k2         # RK2 full step using the midpoint slope

    fig, axes = plt.subplots(1, 3, figsize=(12, 4), sharey=True)
    titles = [r"1. Slope $k_1$ at $t_n$, half step",
              r"2. Slope $k_2$ at the midpoint",
              r"3. Full step with slope $k_2$"]

    for i, ax in enumerate(axes):
        ax.plot(tt, exact(tt), "k", label=r"$\exp(t)$")
        ax.plot(t0, x0, "ko")

        # Step 1: Forward Euler half step using k1
        a = 1 if i == 0 else 0.3
        ax.plot([t0, t0 + dt], [x0, xE], "C0--", alpha=a, label=r"slope $k_1$")
        ax.plot(th, xh, "C0o", alpha=1 if i <= 1 else 0.3)

        # Step 2: evaluate the slope k2 at the estimated midpoint
        if i >= 1:
            a = 1 if i == 1 else 0.3
            ax.plot([t0, t0 + dt], [xh - 0.5*dt*k2, xh + 0.5*dt*k2],
                    "C1--", alpha=a, label=r"slope $k_2$")

        # Step 3: take the full step from x_n using k2
        if i == 2:
            ax.plot([t0, t0 + dt], [x0, x1], "C2-", label=r"RK2 step")
            ax.plot(t0 + dt, x1, "C2o")
            ax.plot(t0 + dt, xE, "C0x", ms=8, label=r"Forward Euler")

        ax.set_title(titles[i])
        ax.set_xlabel(r"$t$")
        ax.legend(loc="upper left")

    axes[0].set_ylabel(r"$x(t)$")
    fig.tight_layout()

plot_RK2_step(lambda x: x, np.exp, 1, 0, 1)

In [ ]:
# As with Euler(), f takes only x as an argument

def RK2(f, x, t, dt, n):
    X = [np.array(x)]
    T = [np.array(t)]
    for _ in range(n):
        k1 = f(X[-1])
        k2 = f(X[-1] + 0.5 * dt * k1)
        X.append(X[-1] + dt * k2)
        T.append(T[-1] + dt)
    return np.array(X), np.array(T)

In [ ]:
# RK2 evaluates f twice per step, so n = 32 has the same cost
# (64 evaluations of f) as the Forward Euler run above

n  = 32
dt = 10 / n
X0 = [0, 0.01]
t0 = 0

X2, T2 = RK2(F, X0, t0, dt, n)
plt.plot(T,  0.01 * np.sin(T), label=r"$0.01\sin(t)$")
plt.plot(T2, X2[:,0], "o-",    label=r"RK2")
plt.xlabel(r"$t$")
plt.ylabel(r"$\theta(t)$")
plt.legend()

To see why RK2 is second-order accurate, we compare its update with
the Taylor expansion of the exact solution.
Expanding $\mathbf{x}(t_{n+1})$ about $t_n$ gives
\begin{align}
  \mathbf{x}(t_{n+1}) = \mathbf{x}_n + \frac{d\mathbf{x}}{dt}\Delta t
  + \frac{1}{2}\frac{d^2\mathbf{x}}{dt^2}\Delta t^2 + \mathcal{O}(\Delta t^3).
\end{align}
The first derivative is simply $d\mathbf{x}/dt = \mathbf{f}$.
Using the chain rule, the second derivative is
\begin{align}
  \frac{d^2\mathbf{x}}{dt^2} = \frac{d\mathbf{f}}{dt}
  = \frac{\partial\mathbf{f}}{\partial t}
  + \frac{\partial\mathbf{f}}{\partial\mathbf{x}}\,\mathbf{f},
\end{align}
where $\partial\mathbf{f}/\partial\mathbf{x}$ is the Jacobian matrix
and all terms are evaluated at $(\mathbf{x}_n, t_n)$.
Hence, the exact solution satisfies
\begin{align}
  \mathbf{x}(t_{n+1}) = \mathbf{x}_n + \mathbf{f}\,\Delta t
  + \frac{1}{2}\left(\frac{\partial\mathbf{f}}{\partial t}
  + \frac{\partial\mathbf{f}}{\partial\mathbf{x}}\,\mathbf{f}\right)\Delta t^2
  + \mathcal{O}(\Delta t^3).
\end{align}

For RK2, $\mathbf{k}_1 = \mathbf{f}(\mathbf{x}_n, t_n) = \mathbf{f}$,
and we Taylor expand $\mathbf{k}_2$ in both of its arguments:
\begin{align}
  \mathbf{k}_2
  = \mathbf{f}\left(\mathbf{x}_n + \frac{\Delta t}{2}\mathbf{k}_1, t_n + \frac{\Delta t}{2}\right)
  = \mathbf{f} + \frac{\Delta t}{2}\left(\frac{\partial\mathbf{f}}{\partial t}
  + \frac{\partial\mathbf{f}}{\partial\mathbf{x}}\,\mathbf{f}\right)
  + \mathcal{O}(\Delta t^2).
\end{align}
Substituting this into the RK2 update
$\mathbf{x}_{n+1} = \mathbf{x}_n + \mathbf{k}_2\,\Delta t$ gives
\begin{align}
  \mathbf{x}_{n+1} = \mathbf{x}_n + \mathbf{f}\,\Delta t
  + \frac{1}{2}\left(\frac{\partial\mathbf{f}}{\partial t}
  + \frac{\partial\mathbf{f}}{\partial\mathbf{x}}\,\mathbf{f}\right)\Delta t^2
  + \mathcal{O}(\Delta t^3).
\end{align}

This agrees with the exact solution through the $\Delta t^2$ term, so
the local error of each step is $\mathcal{O}(\Delta t^3)$.
In comparison, Forward Euler only matches the $\Delta t$ term and has a
local error of $\mathcal{O}(\Delta t^2)$.
To integrate up to a fixed time $t$, we need $n = (t - t_0)/\Delta t$
steps, so the local errors accumulate to a global error of
$n \cdot \mathcal{O}(\Delta t^3) = \mathcal{O}(\Delta t^2)$.
RK2 is therefore second-order accurate, while Forward Euler is
first-order accurate.

In [ ]:
# Again, we can study the convergence of the numerical method

N   = np.array([32, 64, 128, 256, 512, 1024])
E1  = np.array([Err(Euler, n) for n in   N])
E2  = np.array([Err(RK2,   n) for n in   N])

plt.loglog(N, 1/N,    "k--", label=r"$1/n$")
plt.loglog(N, 1/N**2, "k-.", label=r"$1/n^2$")
plt.loglog(N, E1,     "o:",  label=r"Forward Euler")
plt.loglog(N, E2,     "o-",  label=r"RK2")
plt.xlabel(r"$n$")
plt.ylabel(r"$\text{err} = \max|x_\text{numeric} - x|$")
plt.legend()

The convergence plot clearly demonstrates that RK2 achieves
second-order accuracy.
This means that the error decreases quadratically with the step size
$\Delta t$.
For example, doubling the number of steps from 512 to 1024 doubles the
computational cost and reduces the error by a factor of about 4,
compared to only a factor of 2 for Forward Euler.
At 1024 steps, RK2 is about 260 times more accurate than Forward
Euler, even though each RK2 step costs only twice as much.
This significant increase in accuracy suggests that higher-order
methods offer substantial benefits, especially for problems requiring
high precision.
Naturally, this brings us to the question: can we improve the accuracy
even further than what RK2 provides?

## Fourth-Order Runge-Kutta Method

In the second-order Runge-Kutta method (RK2), the midpoint value
$\mathbf{x}_n + (\Delta t/2)\,\mathbf{k}_1$ is obtained using a
simple approximation derived from the Forward Euler method.
Specifically, we first calculate $\mathbf{k}_1$ as:
\begin{align}
  \mathbf{k}_1 = \mathbf{f}(\mathbf{x}_n, t_n)
\end{align}
and then use this to approximate $\mathbf{k}_2$:
\begin{align}
  \mathbf{k}_2 = \mathbf{f}\left(\mathbf{x}_n + \frac{1}{2}\Delta t\,\mathbf{k}_1, t_n + \frac{1}{2}\Delta t\right)
\end{align}
This approach provides a basic estimate for the midpoint.

However, we can improve this estimate by refining our approximation of
the midpoint using another application of the Forward Euler estimate.
We calculate $\mathbf{k}_3$ as follows:
\begin{align}
  \mathbf{k}_3 = \mathbf{f}\left(\mathbf{x}_n + \frac{1}{2}\Delta t\,\mathbf{k}_2, t_n + \frac{1}{2}\Delta t\right)
\end{align}
Finally, to complete the full step with this improved midpoint
estimate, we compute $\mathbf{k}_4$:
\begin{align}
  \mathbf{k}_4 = \mathbf{f}(\mathbf{x}_n + \Delta t\,\mathbf{k}_3, t_n + \Delta t)
\end{align}

In [ ]:
# HANDSON: Use `k4` alone to create an ODE integrator.
#          What is its order?


The values $\mathbf{k}_1$, $\mathbf{k}_2$, $\mathbf{k}_3$, and
$\mathbf{k}_4$ each provide estimates of how $\mathbf{x}$ will change
over one step, though they carry different error terms.
By combining these estimates carefully, we can construct a higher-order
method that effectively cancels out some of the error components.
This technique, when implemented correctly, provides an even more
accurate approximation than any single $\mathbf{k}_i$-estimate alone.

The classical fourth-order Runge-Kutta method (RK4) is a widely used
technique for numerically solving ordinary differential equations with
high accuracy.
This method calculates four intermediate values at each time step,
combining them to achieve fourth-order accuracy.

The process begins by computing four slopes, $\mathbf{k}_1$,
$\mathbf{k}_2$, $\mathbf{k}_3$, and $\mathbf{k}_4$, which represent
different estimates of the derivative over the interval:
\begin{align}
  \mathbf{k}_1 &= \mathbf{f}     (\mathbf{x}_n,                                     t_n                            ) \\
  \mathbf{k}_2 &= \mathbf{f}\left(\mathbf{x}_n + \frac{1}{2}\Delta t\,\mathbf{k}_1, t_n + \frac{1}{2}\Delta t\right) \\
  \mathbf{k}_3 &= \mathbf{f}\left(\mathbf{x}_n + \frac{1}{2}\Delta t\,\mathbf{k}_2, t_n + \frac{1}{2}\Delta t\right) \\
  \mathbf{k}_4 &= \mathbf{f}\left(\mathbf{x}_n +            \Delta t\,\mathbf{k}_3, t_n +            \Delta t\right)
\end{align}
Once these intermediate values are calculated, they are combined to
update $\mathbf{x}$ at the next step.

The weighted average of these values, with coefficients that cancel
higher-order error terms, is assumed to have the generic form
\begin{align}
  \mathbf{x}_{n+1} = \mathbf{x}_n + (a_1 \mathbf{k}_1 + a_2 \mathbf{k}_2 + a_3 \mathbf{k}_3 + a_4 \mathbf{k}_4)\Delta t + \mathcal{O}(\Delta t^5).
\end{align}

But what should be the values of $a_i$?
The idea is the same as in our analysis of RK2: we Taylor expand both
the exact solution and the RK4 update in powers of $\Delta t$, and
choose the coefficients so that the two agree through the
$\Delta t^4$ term.
The local error of each step is then $\mathcal{O}(\Delta t^5)$, and the
global error is $\mathcal{O}(\Delta t^4)$.

The algebra, however, is much more tedious than for RK2.
For a system of ODEs, the higher-order Taylor coefficients contain
several distinct combinations of derivatives of $\mathbf{f}$, and each
of them must be matched separately.
Through $\Delta t^4$, this gives eight order conditions on the
coefficients.
For the stages $\mathbf{k}_1, \dots, \mathbf{k}_4$ defined above, the
solution is
\begin{align}
  (a_1, a_2, a_3, a_4) = \left(\frac{1}{6},\frac{1}{3},\frac{1}{3},\frac{1}{6}\right),
\end{align}
which gives the classical RK4 update
\begin{align}
  \mathbf{x}_{n+1} = \mathbf{x}_n + \frac{\Delta t}{6}\left(\mathbf{k}_1 + 2\mathbf{k}_2 + 2\mathbf{k}_3 + \mathbf{k}_4\right).
\end{align}

For the full derivation, see *Numerical Recipes* Chapter 17.

In [ ]:
# Let's visualize how RK4 builds up the slope for a single step,
# using the same ODE dx/dt = x with x(0) = 1.
# Each slope k_i is drawn as a short segment at the point where it is
# evaluated, and each trial step from x_n is drawn as a dashed line.

def plot_RK4_step(f, exact, x0, t0, dt):
    tt = np.linspace(t0, t0 + dt, 101)
    th = t0 + 0.5 * dt
    t1 = t0 + dt

    k1 = f(x0)               # slope at the beginning of the step
    xa = x0 + 0.5 * dt * k1  # first midpoint estimate, using k1
    k2 = f(xa)               # slope at the first midpoint estimate
    xb = x0 + 0.5 * dt * k2  # second midpoint estimate, using k2
    k3 = f(xb)               # slope at the second midpoint estimate
    xc = x0 + dt * k3        # endpoint estimate, using k3
    k4 = f(xc)               # slope at the endpoint estimate
    k  = (k1 + 2*k2 + 2*k3 + k4) / 6
    x1 = x0 + dt * k         # RK4 full step using the weighted slope

    def slope(ax, t, x, s, c, a=1, label=None):
        h = 0.2 * dt
        ax.plot([t - h, t + h], [x - h*s, x + h*s], c, lw=3, alpha=a, label=label)
        ax.plot(t, x, "o", color=c, alpha=a)

    def trial(ax, t, x, c, a=1):
        ax.plot([t0, t], [x0, x], "--", color=c, alpha=a)

    fig, axes = plt.subplots(2, 2, figsize=(10, 8), sharex=True, sharey=True)
    axes = axes.flatten()
    titles = [r"1. Slope $k_1$ at $t_n$, half step",
              r"2. Slope $k_2$, half step again",
              r"3. Slope $k_3$, full step, slope $k_4$",
              r"4. Full step with weighted slope"]

    for i, ax in enumerate(axes):
        ax.plot(tt, exact(tt), "k", label=r"$\exp(t)$")
        ax.plot(t0, x0, "ko")
        fade = lambda j: 1 if i == j else 0.3

        # Step 1: slope k1 at x_n, half step to the first midpoint estimate
        slope(ax, t0, x0, k1, "C0", fade(0), r"slope $k_1$")
        trial(ax, th, xa, "C0", fade(0))

        # Step 2: slope k2 at the first midpoint estimate, half step again
        if i >= 1:
            slope(ax, th, xa, k2, "C1", fade(1), r"slope $k_2$")
            trial(ax, th, xb, "C1", fade(1))

        # Step 3: slope k3 at the second midpoint estimate, full step,
        # then slope k4 at the endpoint estimate
        if i >= 2:
            slope(ax, th, xb, k3, "C2", fade(2), r"slope $k_3$")
            trial(ax, t1, xc, "C2", fade(2))
            slope(ax, t1, xc, k4, "C3", fade(2), r"slope $k_4$")

        # Step 4: full step using (k1 + 2 k2 + 2 k3 + k4) / 6
        if i == 3:
            ax.plot([t0, t1], [x0, x1], "C4-", label=r"RK4 step")
            ax.plot(t1, x1, "C4o")
            ax.plot(t1, x0 + dt*k1, "kx", ms=8, label=r"Forward Euler")
            ax.plot(t1, x0 + dt*k2, "k+", ms=10, label=r"RK2")

        ax.set_title(titles[i])
        ax.legend(loc="upper left")

    for ax in axes[2:]:
        ax.set_xlabel(r"$t$")
    for ax in axes[::2]:
        ax.set_ylabel(r"$x(t)$")
    fig.tight_layout()

    print(f"k1 = {k1:.4f}, k2 = {k2:.4f}, k3 = {k3:.4f}, k4 = {k4:.4f}")
    print(f"Euler: {x0 + dt*k1:.5f}, RK2: {x0 + dt*k2:.5f}, "
          f"RK4: {x1:.5f}, exact: {exact(t1):.5f}")

plot_RK4_step(lambda x: x, np.exp, 1, 0, 1)

In [ ]:
# We are now ready to implement RK4!
# As before, f takes only x as an argument

def RK4(f, x, t, dt, n):
    X = [np.array(x)]
    T = [np.array(t)]
    for _ in range(n):
        k1 = f(X[-1]                )
        k2 = f(X[-1] + 0.5 * dt * k1)
        k3 = f(X[-1] + 0.5 * dt * k2)
        k4 = f(X[-1] +       dt * k3)
        X.append(X[-1] + dt * (k1/6 + k2/3 + k3/3 + k4/6))
        T.append(T[-1] + dt)
    return np.array(X), np.array(T)

In [ ]:
# RK4 evaluates f four times per step, so n = 16 has the same cost
# (64 evaluations of f) as the Forward Euler and RK2 runs above

n  = 16
dt = 10/n
X0 = [0, 0.01]
t0 = 0

X4, T4 = RK4(F, X0, t0, dt, n)
plt.plot(T, 0.01*np.sin(T), label=r"$0.01\sin(t)$")
plt.plot(T4, X4[:,0], "o-", label=r"RK4")
plt.xlabel(r"$t$")
plt.ylabel(r"$\theta(t)$")
plt.legend()

In [ ]:
# Again, we can study the convergence of the numerical method

N   = np.array([32, 64, 128, 256, 512, 1024])
E1  = np.array([Err(Euler, n) for n in   N])
E2  = np.array([Err(RK2,   n) for n in   N])
E4  = np.array([Err(RK4,   n) for n in   N])

plt.loglog(N, 1/N,    "k--",  label=r"$1/n$")
plt.loglog(N, 1/N**2, "k-.",  label=r"$1/n^2$")
plt.loglog(N, 1/N**4, "k-..", label=r"$1/n^4$")
plt.loglog(N, E1,     "o:",   label=r"Forward Euler")
plt.loglog(N, E2,     "o:",   label=r"RK2")
plt.loglog(N, E4,     "o-",   label=r"RK4")
plt.xlabel(r"$n$")
plt.ylabel(r"$\text{err} = \max|x_\text{numeric} - x|$")
plt.legend()

The classical RK4 is not the only fourth-order Runge-Kutta method.
A well-known alternative is Wilhelm Kutta's (1901) "3/8 rule", which
samples the derivative at $t_n$, $t_n + \Delta t/3$, $t_n + 2\Delta t/3$,
and $t_n + \Delta t$:
\begin{align}
  \mathbf{k}_1 &= \mathbf{f}(\mathbf{x}_n, t_n) \\
  \mathbf{k}_2 &= \mathbf{f}\left(\mathbf{x}_n + \frac{1}{3}\Delta t\,\mathbf{k}_1, t_n + \frac{1}{3}\Delta t\right) \\
  \mathbf{k}_3 &= \mathbf{f}\left(\mathbf{x}_n + \Delta t\left(-\frac{1}{3}\mathbf{k}_1 + \mathbf{k}_2\right), t_n + \frac{2}{3}\Delta t\right) \\
  \mathbf{k}_4 &= \mathbf{f}\left(\mathbf{x}_n + \Delta t\left(\mathbf{k}_1 - \mathbf{k}_2 + \mathbf{k}_3\right), t_n + \Delta t\right)
\end{align}
with the update
\begin{align}
  \mathbf{x}_{n+1} = \mathbf{x}_n + \frac{\Delta t}{8}\left(\mathbf{k}_1 + 3\mathbf{k}_2 + 3\mathbf{k}_3 + \mathbf{k}_4\right).
\end{align}

This suggests that Runge-Kutta methods are really a "family", where
many different choices can be used to construct numerical schemes with
the same order.
The performance of the numerical scheme, nevertheless, depends on the
number of operations as well as the properties of the ODEs being
solved.

In [ ]:
# HANDSON: Implement the "3/8 rule" and compare it to other ODE
#          integrators that we've implemented.


Note that the stages of the 3/8 rule are not simply chained like those
of the classical RK4.
In `RK4()`, each slope $\mathbf{k}_i$ depends only on the previous
slope $\mathbf{k}_{i-1}$.
In the 3/8 rule, $\mathbf{k}_3$ and $\mathbf{k}_4$ depend on a
combination of several previous slopes $\mathbf{k}_j$ with $j < i$.
If we simply keep the chained structure of `RK4()` and only change the
sample times and weights to those of the 3/8 rule, the resulting
algorithm is only third-order accurate.
In general, a Runge-Kutta method is specified by all of its stage
coefficients, which are conveniently summarized in a
[Butcher tableau](https://en.wikipedia.org/wiki/List_of_Runge%E2%80%93Kutta_methods).

## Application: Complete Pendulum Problem

With a highly accurate ODE integrator like RK4, we are now equipped to
solve the complete nonlinear gravity pendulum problem.
No simplifying assumptions are needed.

In [ ]:
def F(x):
    theta, omega = x
    return np.array([omega, -np.sin(theta)])

n  = 1000
dt = 10 / n
X0 = [0, 0.1]
t0 = 0
X4, T4 = RK4(F, X0, t0, dt, n)

Theta4 = X4[:,0]
Omega4 = X4[:,1]

plt.plot(T,  X0[1] * np.sin(T), label=r"$\Omega_0\sin(t)$")
plt.plot(T4, Theta4, ':',       label=r"RK4")
plt.xlabel(r"$t$")
plt.ylabel(r"$\theta(t)$")
plt.legend()

In [ ]:
# When the initial velocity is no longer small, the simple harmonic
# oscillator approximation is no longer valid.

n  = 1000
dt = 10 / n
X0 = [0, 1]
t0 = 0
X4, T4 = RK4(F, X0, t0, dt, n)

Theta4 = X4[:,0]
Omega4 = X4[:,1]

plt.plot(T,  X0[1] * np.sin(T), label=r"$\Omega_0\sin(t)$")
plt.plot(T4, Theta4, ':',       label=r"RK4")
plt.xlabel(r"$t$")
plt.ylabel(r"$\theta(t)$")
plt.legend()

In [ ]:
# When the initial velocity is large enough, the solution is not
# periodic.
# What's going on?

n  = 1000
dt = 10 / n
X0 = [0, 2]
t0 = 0
X4, T4 = RK4(F, X0, t0, dt, n)

Theta4 = X4[:,0]
Omega4 = X4[:,1]

plt.plot(T,  X0[1] * np.sin(T), label=r"$\Omega_0\sin(t)$")
plt.plot(T4, Theta4, ':',    label=r"RK4")
plt.xlabel(r"$t$")
plt.ylabel(r"$\theta(t)$")
plt.legend()

In [ ]:
# HANDSON: Try different values of dt, n, and initial conditions, and
#          observe the results.


## Convergence of Integrators

When solving ordinary differential equations (ODEs) numerically,
errors inevitably enter through truncation (e.g., approximating
derivatives with finite differences) and round-off (finite precision
arithmetic).
If these errors grow uncontrollably, the computed solution may diverge
from the true behavior, even if the method is very accurate for short
times.

It is useful to distinguish between several related ideas:
* Accuracy:
  How small the error is for a given step size $\Delta t$.
  It is usually characterized by the order $p$ of the method:
  a $p$-th order method has error $\mathcal{O}(\Delta t^p)$, so halving
  $\Delta t$ reduces the error by a factor of $2^p$.
* Consistency:
  The numerical method reproduces the original ODE as
  $\Delta t \to 0$:
  writing the update as a difference quotient, e.g.,
  $(x_{n+1} - x_n)/\Delta t = f(x_n)$ for forward Euler, recovers
  $dx/dt = f(x)$ in the limit.
  Formally, the local truncation error $\tau$ (the error made in a
  single step assuming exact input) must vanish faster than
  $\Delta t$, i.e., $\tau / \Delta t \to 0$.
  A method with $\tau = \mathcal{O}(\Delta t^{p+1})$, $p \geq 1$, is
  consistent of order $p$.
* Stability:
  Errors introduced during computation (truncation or round-off) are
  not amplified by the numerical method as steps are repeated.
  There are two distinct notions:
  * Zero-stability:
    small perturbations (e.g., in the initial condition) stay bounded
    as $\Delta t \to 0$ on a fixed time interval $[0, T]$.
    It concerns the limit of small steps, and it is the stability
    needed for convergence (see below).
  * Absolute stability:
    for a *fixed* step size $\Delta t$, the numerical solution of the
    linear test equation $dx/dt = \lambda x$ with
    $\mathrm{Re}(\lambda) \leq 0$ does not grow from step to step.
    It determines how large $\Delta t$ can be in practice (see below).
* Convergence:
  The global error $e_n = |x_n - x(t_n)|$, the accumulated error after
  many steps, vanishes at a fixed final time $T = N \Delta t$ as
  $\Delta t \to 0$.
  A method is convergent of order $p$ if $e_N = \mathcal{O}(\Delta t^p)$.
  Since the local truncation errors accumulate over
  $N = T/\Delta t$ steps, a local truncation error of
  $\mathcal{O}(\Delta t^{p+1})$ typically gives a global error of
  $\mathcal{O}(\Delta t^p)$.

All integrators in these notes are one-step methods:
$x_{n+1}$ is computed from $x_n$ alone (possibly with intermediate
stages, as in Runge-Kutta), without using earlier values
$x_{n-1}, x_{n-2}, \dots$ as multistep methods do.
For such one-step ODE integrators, these concepts are related by a
central result of numerical analysis:
\begin{align}
  \text{Consistency + Zero-stability } \iff \text{ Convergence.}
\end{align}
This is Dahlquist's equivalence theorem, the ODE analogue of the
[Lax Equivalence Theorem](https://en.wikipedia.org/wiki/Lax_equivalence_theorem)
for linear finite difference methods for partial differential
equations.
Every consistent one-step method (including all Runge-Kutta methods in
these notes, as well as the implicit and symplectic methods of the next
lecture) is automatically zero-stable
for well-behaved $f$, so for us:
\begin{align}
  \text{Consistency } \implies \text{ Convergence.}
\end{align}

Convergence, however, only describes the limit $\Delta t \to 0$.
It does not tell us how large $\Delta t$ can be in practice.
For that we study *absolute stability*;
from now on, "stability" in these notes always refers to it.
We use the linear test equation:
\begin{align}
  \frac{dx}{dt} = \lambda x, \quad \lambda \in \mathbb{C},
  \quad \mathrm{Re}(\lambda) \leq 0.
\end{align}
Its exact solution $x(t) = x_0 e^{\lambda t}$ does not grow.
Applying a one-step method produces an update:
\begin{align}
  x_{n+1} = R(z) x_n, \quad z = \lambda \Delta t,
\end{align}
where $R(z)$ is the amplification factor.
For example, forward Euler gives
$x_{n+1} = x_n + \Delta t\, \lambda x_n = (1 + z)\, x_n$, so
$R(z) = 1 + z$.
The method is stable for a given $z$ if
\begin{align}
  |R(z)| \leq 1,
\end{align}
so that the numerical solution, and any error in it, does not grow
from step to step.
The set of $z$ satisfying this condition is the stability region.
Some related terms:
* A method that is stable only for sufficiently small $\Delta t$ is
  conditionally stable.
* A method whose stability region contains the entire left half-plane
  $\mathrm{Re}(z) \leq 0$ is
  [A-stable](https://en.wikipedia.org/wiki/Stiff_equation#A-stability).
* If no $\Delta t > 0$ makes a method stable for a given $\lambda$, it is
  unconditionally unstable for that problem.

### Forward Euler Stability and Consistency

The Forward Euler method for $dx/dt = f(x)$ is
\begin{align}
  x_{n+1} = x_n + \Delta t f(x_n).
\end{align}
Expanding the true solution with a Taylor series, using
$\ddot{x} = \frac{d}{dt} f(x) = f'(x)\, f(x)$:
\begin{align}
  x(t_{n+1}) = x(t_n + \Delta t) = x(t_n) + \Delta t f(x(t_n)) + \frac{1}{2} \Delta t^2 f'(x(t_n)) f(x(t_n)) + \mathcal{O}(\Delta t^3)
\end{align}

The difference between the true solution and one forward Euler step
started from the exact value $x_n = x(t_n)$ is the local truncation
error
\begin{align}
  \tau = \frac{1}{2} \Delta t^2 f' f + \mathcal{O}(\Delta t^3) = \mathcal{O}(\Delta t^2).
\end{align}
Thus $\tau / \Delta t = \mathcal{O}(\Delta t) \to 0$:
forward Euler is consistent of order $p = 1$ (first-order consistent),
and by the result above its global error is $\mathcal{O}(\Delta t)$.

Applying forward Euler to the test equation gives:
\begin{align}
  x_{n+1} = (1 + \lambda \Delta t) x_n,
\end{align}
so the amplification factor is
\begin{align}
  R(z) = 1 + z, \quad z = \lambda \Delta t.
\end{align}

The stability condition requires
\begin{align}
  |1 + z| \leq 1,
\end{align}
which is the closed disk of radius 1 centered at $z = -1$ in the
complex plane.

In [ ]:
# Define grid in complex plane
Re = np.linspace(-3, 3, 601)
Im = np.linspace(-3, 3, 601)

Re, Im = np.meshgrid(Re, Im)
Z = Re + 1j * Im

In [ ]:
# Forward Euler amplification factor

RfE = lambda z: abs(1 + z)

In [ ]:
import matplotlib.patches as mpatches

plt.contour (Re, Im, RfE(Z), levels=[1],   colors=['C0'])
plt.contourf(Re, Im, RfE(Z), levels=[0,1], colors=['C0'], alpha=0.1)

plt.legend(handles=[
    mpatches.Patch(color='C0', label="Forward Euler stability region"),
])

plt.xlabel(r"Re($z$) = Re($\lambda\Delta t$)")
plt.ylabel(r"Im($z$) = Im($\lambda\Delta t$)")
plt.gca().set_aspect('equal')

Combining the two results:
* Convergence:
  forward Euler is first-order convergent, i.e., the global error at a
  fixed time $T$ scales as $\mathcal{O}(\Delta t)$ as $\Delta t \to 0$.
* Stability:
  for a fixed, practical $\Delta t$, we additionally need
  $z = \lambda \Delta t$ inside the stability region.
  Otherwise the numerical solution grows step by step even though the
  exact solution decays.
  As $\Delta t \to 0$, $z$ eventually lies inside the region, which is
  consistent with convergence.

#### Exponential Decay

To illustrate stability and convergence, we reuse the `Euler()`
integrator implemented above to solve the equation $dx/dt = -x$, which
has $\lambda = -1$ and solution $\exp(-t)$.

In [ ]:
f = lambda x: -x
t = 16
T = np.linspace(0, t, t*100+1)
X = np.exp(-T)

n = 32; X11, T11 = Euler(f, x=1, t=0, dt=t/n, n=n)
n = 16; X12, T12 = Euler(f, x=1, t=0, dt=t/n, n=n)
n = 8;  X13, T13 = Euler(f, x=1, t=0, dt=t/n, n=n)
n = 4;  X14, T14 = Euler(f, x=1, t=0, dt=t/n, n=n)

plt.plot(T,   X,   'k',    label=r"$\exp(-t)$")
plt.plot(T11, X11, "C0o-", label=f"Forward Euler with dt={t/32}")
plt.plot(T12, X12, "C1o-", label=f"Forward Euler with dt={t/16}")
plt.plot(T13, X13, "C2o-", label=f"Forward Euler with dt={t/8} (critical stability)")
plt.plot(T14, X14, "C3o-", label=f"Forward Euler with dt={t/4}")
plt.xlabel(r"$t$")
plt.ylabel(r"$x(t)$")
plt.ylim(-10, 10)
plt.legend()

With $\lambda = -1$, we have $z = -\Delta t$, and the forward Euler
amplification factor is $R = 1 - \Delta t$:
* $\Delta t = 0.5$ ($R = 0.5$):
  stable; the solution decays, with an $\mathcal{O}(\Delta t)$ error.
* $\Delta t = 1$ ($R = 0$):
  stable, but the solution drops to exactly zero after one step.
* $\Delta t = 2$ ($R = -1$):
  on the boundary of the stability region (critical stability);
  the solution flips sign every step without decaying, which is
  qualitatively wrong.
* $\Delta t = 4$ ($R = -3$):
  unstable; the solution oscillates with exponentially growing
  amplitude.

In general, forward Euler is stable for real $\lambda < 0$ only if
$\Delta t \leq 2/|\lambda|$.
As $\Delta t \to 0$, the error decreases, as guaranteed by
convergence.

This suggests that,
* Consistency:
  Forward Euler has local truncation error $\mathcal{O}(\Delta t^2)$,
  i.e., it is first-order consistent.
* Convergence:
  Being a consistent one-step method, it converges with global error
  $\mathcal{O}(\Delta t)$.
* Stability:
  Its stability region is the disk $|1 + z| \leq 1$.

Implications:
* For real $\lambda < 0$ (decaying solutions), stability requires
  $\Delta t \leq 2/|\lambda|$.
* For stiff systems, where $|\lambda|$ is very large, this step size
  restriction is too severe to be practical.
* For oscillatory systems, $\lambda$ is imaginary, and forward Euler is
  unstable for every $\Delta t > 0$ (see the harmonic oscillator
  below).

This motivates the need for implicit integrators (backward Euler,
implicit midpoint) and symplectic integrators for Hamiltonian systems,
which we will study in the next lecture.

#### Simple Harmonic Oscillator

The exponential decay problem is exactly the linear test equation and
is a good illustrative example, but many physical systems of interest are
naturally written as systems of coupled ODEs.

Recalling from earlier in this lecture, an important example is the
simple harmonic oscillator:
\begin{align}
  \frac{d}{dt}
  \begin{bmatrix}
    \theta(t) \\
    \Omega(t)
  \end{bmatrix}
  =
  \begin{bmatrix}
    0    & 1 \\
    -g/l & 0
  \end{bmatrix}
  \begin{bmatrix}
    \theta(t) \\
    \Omega(t)
  \end{bmatrix}.
\end{align}
Here $\theta(t)$ is the angular displacement, $\Omega(t)$ the angular
velocity, and $g/l$ sets the natural frequency $\omega_0^2 = g/l$.


Applying the Forward Euler method gives:
\begin{align}
  \begin{bmatrix}
    \theta_{n+1} \\
    \Omega_{n+1}
  \end{bmatrix}
  =
  \begin{bmatrix}
    \theta_{n} \\
    \Omega_{n}
  \end{bmatrix}
  +
  \Delta t
  \begin{bmatrix}
    0 & 1 \\
    -g/l & 0
  \end{bmatrix}
  \begin{bmatrix}
    \theta_{n} \\
    \Omega_{n}
  \end{bmatrix}.
\end{align}

This can be written compactly as:
\begin{align}
  \mathbf{x}_{n+1} = A \mathbf{x}_n, \quad
  A =
  \begin{bmatrix}
    1              & \Delta t \\
    -(g/l)\Delta t & 1
  \end{bmatrix}.
\end{align}

Here $A$ is the amplification matrix, the multi-dimensional analogue
of the scalar amplification factor $R(z)$.

What should we expect of the numerical solution?
The original ODE matrix
\begin{align}
  B =
  \begin{bmatrix}
    0    & 1 \\
    -g/l & 0
  \end{bmatrix},
  \quad
  \det(B - \lambda I) = \lambda^2 + \frac{g}{l} = 0
  \implies
  \lambda = \pm i \omega_0, \quad \omega_0 = \sqrt{\frac{g}{l}},
\end{align}
has purely imaginary eigenvalues, $\mathrm{Re}(\lambda) = 0$.
This is the boundary case $\mathrm{Re}(\lambda) = 0$ of the test
equation's assumption $\mathrm{Re}(\lambda) \leq 0$:
the exact solution neither grows nor decays, but oscillates with
constant amplitude.
(In general, dissipative systems have $\mathrm{Re}(\lambda) < 0$,
undamped oscillations have $\mathrm{Re}(\lambda) = 0$, and if some
$\mathrm{Re}(\lambda) > 0$ the exact solution itself grows in that
direction.)
Since the exact solution is bounded, stability means that the
numerical solution should stay bounded too.
Because $\mathbf{x}_n = A^n \mathbf{x}_0$, each eigen-component of
$\mathbf{x}_0$ is multiplied by the corresponding eigenvalue $\mu$ of
$A$ at every step, so stability requires $|\mu| \leq 1$ for all
eigenvalues.
(We write $\mu$ to avoid confusion with $\lambda$ in the test
equation.)

This is the test equation in disguise:
diagonalizing $B$ splits the system into independent scalar equations
$dy/dt = \lambda y$, one for each eigenvalue $\lambda = \pm i\omega_0$
of $B$, and the eigenvalues of $A$ are then $\mu = R(\lambda \Delta t)$.
We verify this below.

Let's compute them.
The characteristic polynomial of $A$ is:
\begin{align}
  \det(A - \mu I) =
  \begin{vmatrix}
  1 - \mu        & \Delta t \\
  -(g/l)\Delta t & 1 - \mu
  \end{vmatrix}
  = (1-\mu)^2 + \frac{g}{l} \Delta t^2.
\end{align}

Setting this to zero, $(1 - \mu)^2 = -(g/l)\Delta t^2$.
Thus the eigenvalues are:
\begin{align}
  \mu_{\pm} = 1 \pm i \omega_0 \Delta t, \quad \omega_0 = \sqrt{\frac{g}{l}}.
\end{align}

These are exactly the forward Euler amplification factor
$R(z) = 1 + z$ evaluated at $z = \pm i \omega_0 \Delta t$, because
$\pm i\omega_0$ are the eigenvalues of the original ODE matrix.
In other words, the harmonic oscillator is the test equation with
purely imaginary $\lambda = \pm i\omega_0$, so its $z$ lies on the
imaginary axis.

The magnitude of each eigenvalue is:
\begin{align}
  |\mu_{\pm}| = \sqrt{1 + \omega_0^2 \Delta t^2}.
\end{align}
This is always larger than 1 for any nonzero time step:
no point of the imaginary axis other than $z = 0$ lies inside the
forward Euler stability disk (see the plot below).
Therefore, forward Euler applied to the harmonic oscillator is
unconditionally unstable.

In [ ]:
# Mark z = ±iω₀Δt of the harmonic oscillator on top of the forward Euler
# stability region: the points lie on the imaginary axis, outside the disk.
# Unlike earlier (g = l = 1), we now use g = 9.8 and l = 1.
g, l   = 9.8, 1
omega0 = np.sqrt(g/l)  # natural frequency omega_0

plt.contour (Re, Im, RfE(Z), levels=[1],   colors=['C0'])
plt.contourf(Re, Im, RfE(Z), levels=[0,1], colors=['C0'], alpha=0.1)

handles = [mpatches.Patch(color='C0', label="Forward Euler stability region")]
for dt, color in [(0.1, 'C1'), (0.5, 'C3')]:
    h, = plt.plot([0, 0], [omega0*dt, -omega0*dt], 'x', color=color, ms=10,
                  label=rf"$z = \pm i\omega_0\Delta t$, $\Delta t = {dt}$")
    handles.append(h)
plt.legend(handles=handles)

plt.xlabel(r"Re($z$) = Re($\lambda\Delta t$)")
plt.ylabel(r"Im($z$) = Im($\lambda\Delta t$)")
plt.gca().set_aspect('equal')

In [ ]:
# Parameters (g and l are set above)
# Start from rest at theta_0 = 1, so theta(t) = cos(omega_0 t)
t, n = 100, 10000

def Sho(x):
    theta, omega = x
    return np.array([omega, -(g/l) * theta])

def sho_solution(t):
    return np.cos(np.sqrt(g/l)*t)

T     = np.linspace(0, t, t*100+1)
Theta = sho_solution(T)

X1, T1 = Euler(Sho, x=[1.0, 0.0], t=0, dt=t/n, n=n)
Theta1, Omega1 = X1[:,0], X1[:,1]

plt.plot(T,  Theta, 'k', label=r"Exact $\theta(t)$")
plt.plot(T1, Theta1,     label=r"Forward Euler $\theta(t)$")
plt.xlabel(r"$t$")
plt.ylabel(r"$\theta(t)$")
plt.ylim(-10, 10)
plt.legend()

In [ ]:
# HANDSON: try increase the number of time step per period.
#          Does this stop the oscillator from blowing up?


The true solution of the harmonic oscillator is bounded and
oscillatory.
However, the forward Euler amplification matrix has eigenvalues with
modulus larger than 1.
This means that the numerical solution grows in amplitude by a factor
$\sqrt{1 + \omega_0^2 \Delta t^2}$ every step while also oscillating.

Over a fixed time $T = N \Delta t$, the total growth is
$(1 + \omega_0^2 \Delta t^2)^{N/2} \approx e^{\omega_0^2 \Delta t\, T / 2}$,
which tends to 1 as $\Delta t \to 0$, so forward Euler still converges,
as it must.
But for any fixed $\Delta t$, no matter how small, the amplitude
eventually blows up if we integrate long enough.
Reducing $\Delta t$ only delays the blow-up.

In [ ]:
# HANDSON: try plotting the energy of the oscillator.


For the harmonic oscillator, forward Euler fails to preserve the
bounded, energy-conserving nature of the physics.
Since the oscillator's $z = \pm i \omega_0 \Delta t$ lies on the
imaginary axis, the long-term behavior of a method is set by $|R(z)|$
there:
* $|R| > 1$ (e.g., forward Euler): artificial growth of energy.
* $|R| < 1$ (e.g., backward Euler, next lecture): artificial damping of
  energy.
* $|R| = 1$ (e.g., implicit midpoint, next lecture): the amplitude is
  preserved.

Symplectic integrators (e.g., leapfrog, next lecture) go further and
preserve the geometric structure of Hamiltonian dynamics, producing
qualitatively correct long-term behavior.

This example connects stability analysis directly to physics:
the choice of integrator is not just about error size, but about
preserving the qualitative features of the dynamics.

## Final Comments

Building on the insights from numerical integration of functions, we
developed multiple ODE integrators.
Specifically, we introduced the 4th-order Runge-Kutta (RK4) method as a
powerful and effective algorithm for solving ordinary differential
equations.

The classical RK4 method is not only highly accurate but also robust
and relatively simple to implement.
When paired with advanced techniques such as adaptive time-stepping
and dense output, `RK4` becomes even more versatile.
This makes `RK4` a reliable "workhorse" for ODE integration across
many applications.

However, it's important to remember that there are many other ODE
integrators available, each with strengths suited to specific types of
equations.
For certain problems, especially those involving stiffness or
requiring highly efficient, precise computations, other methods may
offer better performance.
For more information, see resources like
*Numerical Recipes*
for additional options and implementation guidance.